# Tree-Based Machine Learning Models for Imbalanced Data


## Objective

In this assignment, you will apply tree-based machine learning models to the CDC Diabetes Health Indicators dataset to predict diabetes diagnosis. 
You will experiment with different classifiers, account for class imbalance, and evaluate models using appropriate metrics.


## Data Acquisition
Fetch the CDC Diabetes Health Indicators dataset and inspect it.

In [3]:
from ucimlrepo import fetch_ucirepo

# Fetch dataset
cdc_diabetes_health_indicators = fetch_ucirepo(id=891)

# Extract features and target
X = cdc_diabetes_health_indicators.data.features
y = cdc_diabetes_health_indicators.data.targets

# Show metadata and variable info
print(cdc_diabetes_health_indicators.metadata)
print(cdc_diabetes_health_indicators.variables)

{'uci_id': 891, 'name': 'CDC Diabetes Health Indicators', 'repository_url': 'https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators', 'data_url': 'https://archive.ics.uci.edu/static/public/891/data.csv', 'abstract': 'The Diabetes Health Indicators Dataset contains healthcare statistics and lifestyle survey information about people in general along with their diagnosis of diabetes. The 35 features consist of some demographics, lab test results, and answers to survey questions for each patient. The target variable for classification is whether a patient has diabetes, is pre-diabetic, or healthy. ', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Tabular', 'Multivariate'], 'num_instances': 253680, 'num_features': 21, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Sex', 'Age', 'Education Level', 'Income'], 'target_col': ['Diabetes_binary'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_

In [4]:
positive_ratio = y.value_counts(normalize=True) * 100
print("Percentage of each class in y:\n", positive_ratio)

Percentage of each class in y:
 Diabetes_binary
0                  86.066698
1                  13.933302
Name: proportion, dtype: float64


## Train-Test Split
Split the data into training (70%) and test (30%) sets.

In [5]:
from sklearn.model_selection import train_test_split

# Complete the following:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print("Shape of training data is: ", X_train.shape)

Shape of training data is:  (177576, 21)


## Model Imports
Import the models and metrics you'll use.

In [6]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight

## Model Training and Evaluation
You will train four models:

- XGBoost (https://xgboost.readthedocs.io/en/stable/get_started.html)
- Decision Tree (https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html)
- Random Forest (https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- AdaBoost (https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html)

### For each model:
1. Experiment with different hyperparameter settings.
2. Since the dataset is imbalanced:
- Use `class_weight='balanced'` for Random Forest and Decision Tree.
- Use `scale_pos_weight` for XGBoost.
- Use sample weights for AdaBoost (e.g., with `compute_sample_weight`).

3. Report the PR-AUC (`average_precision_score`) and confusion matrix (`confusion_matrix`) for each model and each hyperparameter combination.
4. Additionally, for the Decision Tree, also report the PR-AUC and confusion matrix without using `class_weight` and any hyperparameter tuning. Describe how the metrics change compared to the tuned model.
5. Which model and hyperparameter setting gives the best PR-AUC and confusion matrix. 

### XGBoost Model

In [7]:
# Define and train the model

negative_count = (y_train == 0).sum().item()
positive_count = (y_train == 1).sum().item()

scale_pos_weight = negative_count / positive_count

xgb_params = [
    {"n_estimators": 100, "max_depth": 3, "learning_rate": 0.1},
    {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.1},
    {"n_estimators": 200, "max_depth": 5, "learning_rate": 0.05},
]

xgb_results = []

for params in xgb_params:

    xgb_model = XGBClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        scale_pos_weight=scale_pos_weight,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )

    xgb_model.fit(X_train, y_train.values.ravel())

    # Predict and evaluate

    y_pred = xgb_model.predict(X_test)
    y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]

    pr_auc = average_precision_score(y_test, y_pred_proba)
    cm = confusion_matrix(y_test, y_pred)

    xgb_results.append((params, pr_auc, cm))

    print("\nXGBoost Parameters:", params)
    print("PR-AUC:", pr_auc)
    print("Confusion Matrix:")
    print(cm)

# Best XGBoost setting
best_xgb = max(xgb_results, key=lambda x: x[1])

print("\nBest XGBoost Setting:")
print(best_xgb[0])
print("Best PR-AUC:", best_xgb[1])
print("Confusion Matrix:")
print(best_xgb[2])


XGBoost Parameters: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1}
PR-AUC: 0.4244737718188083
Confusion Matrix:
[[46377 19123]
 [ 2163  8441]]

XGBoost Parameters: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.1}
PR-AUC: 0.42686917631559784
Confusion Matrix:
[[46625 18875]
 [ 2239  8365]]

XGBoost Parameters: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05}
PR-AUC: 0.42742233659840323
Confusion Matrix:
[[46406 19094]
 [ 2164  8440]]

Best XGBoost Setting:
{'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.05}
Best PR-AUC: 0.42742233659840323
Confusion Matrix:
[[46406 19094]
 [ 2164  8440]]


### Decision Tree Model

In [8]:
# Define and train the model

# Required baseline: no class_weight and no hyperparameter tuning

dt_baseline = DecisionTreeClassifier(random_state=42)

dt_baseline.fit(X_train, y_train.values.ravel())

# Predict and evaluate

y_pred_baseline = dt_baseline.predict(X_test)
y_pred_proba_baseline = dt_baseline.predict_proba(X_test)[:, 1]

baseline_pr_auc = average_precision_score(y_test, y_pred_proba_baseline)

baseline_cm = confusion_matrix(y_test, y_pred_baseline)

print("Decision Tree Baseline")
print("PR-AUC:", baseline_pr_auc)
print("Confusion Matrix:")
print(baseline_cm)


# Tuned Decision Tree settings

dt_params = [
    {"max_depth": 5, "min_samples_leaf": 10},
    {"max_depth": 10, "min_samples_leaf": 10},
    {"max_depth": 15, "min_samples_leaf": 20},
]

dt_results = []

for params in dt_params:

    dt_model = DecisionTreeClassifier(
        max_depth=params["max_depth"],
        min_samples_leaf=params["min_samples_leaf"],
        class_weight="balanced",
        random_state=42,
    )

    dt_model.fit(X_train, y_train.values.ravel())

    # Predict and evaluate

    y_pred = dt_model.predict(X_test)
    y_pred_proba = dt_model.predict_proba(X_test)[:, 1]

    pr_auc = average_precision_score(y_test, y_pred_proba)
    cm = confusion_matrix(y_test, y_pred)

    dt_results.append((params, pr_auc, cm))

    print("\nDecision Tree Parameters:", params)
    print("PR-AUC:", pr_auc)
    print("Confusion Matrix:")
    print(cm)


# Best tuned Decision Tree
best_dt = max(dt_results, key=lambda x: x[1])

print("\nBest Tuned Decision Tree Setting:")
print(best_dt[0])
print("Best PR-AUC:", best_dt[1])
print("Confusion Matrix:")
print(best_dt[2])


# Compare baseline with best tuned model

print("\nComparison with Baseline:")
print("Baseline PR-AUC:", baseline_pr_auc)
print("Tuned PR-AUC:", best_dt[1])
print("PR-AUC Change:", best_dt[1] - baseline_pr_auc)

print("\nBaseline Confusion Matrix:")
print(baseline_cm)

print("\nTuned Confusion Matrix:")
print(best_dt[2])

Decision Tree Baseline
PR-AUC: 0.1918044434740746
Confusion Matrix:
[[57099  8401]
 [ 7096  3508]]

Decision Tree Parameters: {'max_depth': 5, 'min_samples_leaf': 10}
PR-AUC: 0.36564747667190883
Confusion Matrix:
[[46046 19454]
 [ 2554  8050]]

Decision Tree Parameters: {'max_depth': 10, 'min_samples_leaf': 10}
PR-AUC: 0.39879386492363755
Confusion Matrix:
[[45979 19521]
 [ 2410  8194]]

Decision Tree Parameters: {'max_depth': 15, 'min_samples_leaf': 20}
PR-AUC: 0.3698128818532832
Confusion Matrix:
[[46012 19488]
 [ 2827  7777]]

Best Tuned Decision Tree Setting:
{'max_depth': 10, 'min_samples_leaf': 10}
Best PR-AUC: 0.39879386492363755
Confusion Matrix:
[[45979 19521]
 [ 2410  8194]]

Comparison with Baseline:
Baseline PR-AUC: 0.1918044434740746
Tuned PR-AUC: 0.39879386492363755
PR-AUC Change: 0.20698942144956295

Baseline Confusion Matrix:
[[57099  8401]
 [ 7096  3508]]

Tuned Confusion Matrix:
[[45979 19521]
 [ 2410  8194]]


### Random Forest Model

In [9]:
# Define and train the model

rf_params = [
    {"n_estimators": 100, "max_depth": 10, "min_samples_leaf": 5},
    {"n_estimators": 200, "max_depth": 10, "min_samples_leaf": 5},
    {"n_estimators": 200, "max_depth": 15, "min_samples_leaf": 10},
]

rf_results = []

for params in rf_params:

    rf_model = RandomForestClassifier(
        n_estimators=params["n_estimators"],
        max_depth=params["max_depth"],
        min_samples_leaf=params["min_samples_leaf"],
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )

    rf_model.fit(X_train, y_train.values.ravel())

    # Predict and evaluate

    y_pred = rf_model.predict(X_test)
    y_pred_proba = rf_model.predict_proba(X_test)[:, 1]

    pr_auc = average_precision_score(y_test, y_pred_proba)
    cm = confusion_matrix(y_test, y_pred)

    rf_results.append((params, pr_auc, cm))

    print("\nRandom Forest Parameters:", params)
    print("PR-AUC:", pr_auc)
    print("Confusion Matrix:")
    print(cm)


# Best Random Forest setting
best_rf = max(rf_results, key=lambda x: x[1])

print("\nBest Random Forest Setting:")
print(best_rf[0])
print("Best PR-AUC:", best_rf[1])
print("Confusion Matrix:")
print(best_rf[2])


Random Forest Parameters: {'n_estimators': 100, 'max_depth': 10, 'min_samples_leaf': 5}
PR-AUC: 0.4193080709064738
Confusion Matrix:
[[46919 18581]
 [ 2344  8260]]

Random Forest Parameters: {'n_estimators': 200, 'max_depth': 10, 'min_samples_leaf': 5}
PR-AUC: 0.41985588008392044
Confusion Matrix:
[[46887 18613]
 [ 2348  8256]]

Random Forest Parameters: {'n_estimators': 200, 'max_depth': 15, 'min_samples_leaf': 10}
PR-AUC: 0.42407286428044527
Confusion Matrix:
[[47930 17570]
 [ 2483  8121]]

Best Random Forest Setting:
{'n_estimators': 200, 'max_depth': 15, 'min_samples_leaf': 10}
Best PR-AUC: 0.42407286428044527
Confusion Matrix:
[[47930 17570]
 [ 2483  8121]]


### AdaBoost Model

In [10]:
# Define and train the model

sample_weights = compute_sample_weight(
    class_weight="balanced", y=y_train.values.ravel()
)

ada_params = [
    {"n_estimators": 50, "learning_rate": 0.5},
    {"n_estimators": 100, "learning_rate": 0.5},
    {"n_estimators": 100, "learning_rate": 1.0},
]

ada_results = []

for params in ada_params:

    ada_model = AdaBoostClassifier(
        n_estimators=params["n_estimators"],
        learning_rate=params["learning_rate"],
        random_state=42,
    )

    ada_model.fit(X_train, y_train.values.ravel(), sample_weight=sample_weights)

    # Predict and evaluate

    y_pred = ada_model.predict(X_test)
    y_pred_proba = ada_model.predict_proba(X_test)[:, 1]

    pr_auc = average_precision_score(y_test, y_pred_proba)
    cm = confusion_matrix(y_test, y_pred)

    ada_results.append((params, pr_auc, cm))

    print("\nAdaBoost Parameters:", params)
    print("PR-AUC:", pr_auc)
    print("Confusion Matrix:")
    print(cm)


# Best AdaBoost setting
best_ada = max(ada_results, key=lambda x: x[1])

print("\nBest AdaBoost Setting:")
print(best_ada[0])
print("Best PR-AUC:", best_ada[1])
print("Confusion Matrix:")
print(best_ada[2])


AdaBoost Parameters: {'n_estimators': 50, 'learning_rate': 0.5}
PR-AUC: 0.40420034908731606
Confusion Matrix:
[[47323 18177]
 [ 2499  8105]]

AdaBoost Parameters: {'n_estimators': 100, 'learning_rate': 0.5}
PR-AUC: 0.4090767253262894
Confusion Matrix:
[[47104 18396]
 [ 2414  8190]]

AdaBoost Parameters: {'n_estimators': 100, 'learning_rate': 1.0}
PR-AUC: 0.4135199734789121
Confusion Matrix:
[[47853 17647]
 [ 2529  8075]]

Best AdaBoost Setting:
{'n_estimators': 100, 'learning_rate': 1.0}
Best PR-AUC: 0.4135199734789121
Confusion Matrix:
[[47853 17647]
 [ 2529  8075]]


## Results and Comparison

The four models were evaluated using PR-AUC and confusion matrices because the dataset is imbalanced.

| Model | Best Hyperparameters | PR-AUC |
|---|---|---:|
| **XGBoost** | `n_estimators=200, max_depth=5, learning_rate=0.05` | **0.4274** |
| Random Forest | `n_estimators=200, max_depth=15, min_samples_leaf=10` | 0.4241 |
| AdaBoost | `n_estimators=100, learning_rate=1.0` | 0.4135 |
| Decision Tree | `max_depth=10, min_samples_leaf=10` | 0.3988 |

### Best Model

XGBoost achieved the highest PR-AUC of **0.4274** using `n_estimators=200`, `max_depth=5`, and `learning_rate=0.05`.

Its confusion matrix was:

```text
[[46406 19094]
 [ 2164  8440]]


## Evaluation Metrics

### Why PR-AUC and Confusion Matrix Are Better Than Accuracy for Imbalanced Data

Accuracy can be misleading when classes are imbalanced. For example, if only 16% of patients have diabetes, a model that always predicts 'no diabetes' achieves 84% accuracy without actually identifying any diabetic patients.

- **PR-AUC (Precision-Recall Area Under Curve):** Focuses on the minority (positive) class. It's more informative when the positive class is rare.
- **Confusion Matrix:** Shows true positives, false positives, true negatives, and false negatives, allowing you to understand performance beyond a single number.

These metrics provide a more realistic and nuanced view of your model's effectiveness.
